# AP FRQ Auto-Grader

Grade a student's handwritten AP free-response answers against the official marking scheme.

**Pipeline:** question PDF + answer PDF + marking-scheme PDF → Gemini handwriting OCR → marking-scheme parse (cached) → per-question grading → scorecard with per-rubric-point rationale.

**V1 target:** AP Calculus BC. Subject-agnostic by design (see `config.py:SUBJECT_GRADING_ADDENDA`).

## Setup

1. `pip install -r requirements.txt` (or `uv pip install -r requirements.txt`).
2. Copy `.env.example` → `.env`. Pick ONE auth method:
   - **AI Studio API key (recommended):** paste your key into `GEMINI_API_KEY=`. Get one at https://aistudio.google.com/apikey.
   - **Vertex AI service account:** save the JSON to disk and set `GOOGLE_APPLICATION_CREDENTIALS` + `GOOGLE_CLOUD_PROJECT`.
3. Place three PDFs in `Grader/data/`:
   - `questions.pdf` — the typed exam questions
   - `answers.pdf` — the student's scanned handwritten responses
   - `marking-scheme.pdf` — the typed rubric / model answers
4. Run all cells.

In [ ]:
%load_ext autoreload
%autoreload 2

from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Markdown, display

from helpers import (
    render_pdf_to_images,
    get_gemini_client,
    ocr_submission,
    load_rubric,
    grade_question,
)
from schemas import Scorecard
from config import SUBJECT_GRADING_ADDENDA

load_dotenv()  # reads Grader/.env

In [ ]:
CONFIG = {
    # Metadata
    "subject":              "AP Calculus BC",
    "year":                 2024,
    "set":                  None,           # "Set 1" / "Set 2" / None
    "questions":            "all",          # or ["1", "3"]

    # Inputs (three PDFs per exam)
    "questions_pdf":        Path("data/questions.pdf"),       # typed exam questions
    "answers_pdf":          Path("data/answers.pdf"),         # scanned student handwriting
    "marking_scheme_pdf":   Path("data/marking-scheme.pdf"),  # typed rubric / model answers

    # Models
    "model_ocr":            "gemini-2.5-pro",
    "model_grading":        "gemini-2.5-pro",
    "ocr_dpi":              300,
    "rubric_dpi":           200,            # typed text needs less than handwriting

    # Output
    "output_dir":           Path("out/"),

    # Confidence handling (no human checkpoint, so we surface these in the scorecard)
    "low_confidence_threshold": 0.75,
    "fail_on_low_confidence":   False,
}

CONFIG

## Phase 0 — OCR (handwriting → per-question transcripts)

Render the question + answer PDFs and run Gemini joint OCR. Joint pass means Gemini uses the canonical question IDs from the question PDF when labeling each transcribed answer — no separate segmentation step.

Visual validation: the side-by-side review cell below shows each answer page next to its transcript. ⚠️ marks low-confidence questions.

In [ ]:
for label, path in [("questions", CONFIG["questions_pdf"]), ("answers", CONFIG["answers_pdf"])]:
    if not Path(path).exists():
        raise FileNotFoundError(
            f"{label}_pdf not found at {Path(path).resolve()}. "
            f"Place the PDF there or update CONFIG['{label}_pdf']."
        )

question_images = render_pdf_to_images(CONFIG["questions_pdf"], dpi=CONFIG["ocr_dpi"])
answer_images   = render_pdf_to_images(CONFIG["answers_pdf"],   dpi=CONFIG["ocr_dpi"])

print(f"Question PDF: {len(question_images)} page(s) at {CONFIG['ocr_dpi']} DPI")
print(f"Answer PDF:   {len(answer_images)} page(s) at {CONFIG['ocr_dpi']} DPI")

answer_images[0]

In [ ]:
client = get_gemini_client()

submission = ocr_submission(
    client=client,
    question_images=question_images,
    answer_images=answer_images,
    prompt_path=Path("prompts/ocr.txt"),
    model=CONFIG["model_ocr"],
)

print(f"Pages (Answer PDF):  {submission.page_count}")
print(f"Answers extracted:   {len(submission.answers)}")
print(f"Question IDs found:  {[a.question_id for a in submission.answers]}")
print(f"Unassigned chunks:   {len(submission.unassigned_text)}")

### Side-by-side visual review

Each answer-PDF page next to the transcript(s) Gemini mapped to it. Scroll through and confirm each transcript plausibly matches the handwriting. ⚠️ flags questions where confidence is below `CONFIG['low_confidence_threshold']`.

In [ ]:
page_to_answers: dict[int, list] = {}
for ans in submission.answers:
    for p in ans.source_pages:
        page_to_answers.setdefault(p, []).append(ans)

threshold = CONFIG["low_confidence_threshold"]

for pi, img in enumerate(answer_images, start=1):
    display(Markdown(f"### Answer PDF — Page {pi}/{len(answer_images)}"))
    display(img)
    answers_here = page_to_answers.get(pi, [])
    if not answers_here:
        display(Markdown("_(no answers were mapped to this page)_"))
        continue
    for ans in answers_here:
        emoji = "\u2705" if ans.confidence >= threshold else "\u26a0\ufe0f"
        body = (
            f"**Q {ans.question_id}** {emoji} confidence={ans.confidence:.2f} "
            f"(pages {ans.source_pages})\n\n```\n{ans.transcript}\n```"
        )
        if ans.low_confidence_snippets:
            body += "\n\n**Low-confidence snippets:** " + ", ".join(
                f"`{s}`" for s in ans.low_confidence_snippets
            )
        display(Markdown(body))

if submission.unassigned_text:
    display(Markdown("### Unassigned text"))
    for u in submission.unassigned_text:
        display(Markdown(f"- `{u}`"))

## Phase 1 — Marking-scheme parse (cached)

Parse the typed marking-scheme PDF into a structured `ParsedRubric`. First run hits Gemini; subsequent runs load the `.parsed.json` sidecar next to the PDF (free + instant).

To force a fresh parse: delete the `*.parsed.json` file or pass `force_reparse=True`.

In [ ]:
rubric = load_rubric(
    client=client,
    marking_scheme_pdf=CONFIG["marking_scheme_pdf"],
    subject=CONFIG["subject"],
    year=CONFIG["year"],
    set_label=CONFIG["set"],
    prompt_path=Path("prompts/rubric_extract.txt"),
    model=CONFIG["model_ocr"],
    dpi=CONFIG["rubric_dpi"],
)

print(f"Subject:       {rubric.subject}")
print(f"Year:          {rubric.year}")
print(f"Set:           {rubric.set_label}")
print(f"Total points:  {rubric.total_points}")
print(f"Questions:     {len(rubric.questions)}")
for q in rubric.questions:
    print(f"  - Q{q.question_id}: {len(q.rubric_points)} points (max {q.max_points})  — {q.prompt_summary[:80]}")
if rubric.parse_warnings:
    print("\nParse warnings:")
    for w in rubric.parse_warnings:
        print(f"  - {w}")

## Phase 2 — Grade each question

For each question that exists in both the parsed rubric AND the OCR'd answers, call Gemini with `(rubric, transcript)` and parse a `QuestionScorecard`. Subject-specific guidance from `config.py:SUBJECT_GRADING_ADDENDA` is injected.

Any question whose OCR confidence is below threshold gets `review_recommended=True` on all its point scores after grading.

In [ ]:
rubric_by_qid = {q.question_id: q for q in rubric.questions}
answer_by_qid = {a.question_id: a for a in submission.answers}

if CONFIG["questions"] == "all":
    qids_to_grade = sorted(set(rubric_by_qid) & set(answer_by_qid))
else:
    qids_to_grade = CONFIG["questions"]

addendum = SUBJECT_GRADING_ADDENDA.get(CONFIG["subject"], "")

question_scorecards = []
for qid in qids_to_grade:
    qr = rubric_by_qid.get(qid)
    ans = answer_by_qid.get(qid)
    if qr is None:
        print(f"Skipping {qid}: not in parsed rubric")
        continue
    if ans is None:
        print(f"Skipping {qid}: no student answer found")
        continue
    print(f"Grading Q{qid}...")
    qs = grade_question(
        client=client,
        question_rubric=qr,
        answer=ans,
        subject=CONFIG["subject"],
        prompt_path=Path("prompts/grade_question.txt"),
        subject_addendum=addendum,
        model=CONFIG["model_grading"],
        review_recommended=(ans.confidence < CONFIG["low_confidence_threshold"]),
    )
    question_scorecards.append(qs)

total_earned   = sum(qs.points_earned   for qs in question_scorecards)
total_possible = sum(qs.points_possible for qs in question_scorecards)
percentage     = (total_earned / total_possible * 100.0) if total_possible else 0.0

review_flags = []
for qs in question_scorecards:
    if any(ps.review_recommended for ps in qs.point_scores):
        review_flags.append(f"Q{qs.question_id}: OCR confidence below threshold \u2014 verify transcript")
    if any(ps.grading_confidence == "low" for ps in qs.point_scores):
        review_flags.append(f"Q{qs.question_id}: one or more rubric points scored with low confidence")

scorecard = Scorecard(
    subject=CONFIG["subject"],
    year=CONFIG["year"],
    set_label=CONFIG["set"],
    total_points_earned=total_earned,
    total_points_possible=total_possible,
    percentage=percentage,
    questions=question_scorecards,
    review_flags=review_flags,
    generated_at=datetime.now(timezone.utc).isoformat(),
    config_echo={k: str(v) if hasattr(v, "__fspath__") else v for k, v in CONFIG.items()},
)

CONFIG["output_dir"].mkdir(parents=True, exist_ok=True)
subj_slug = CONFIG["subject"].lower().replace(" ", "-")
json_path = CONFIG["output_dir"] / f"scorecard-{subj_slug}-{CONFIG['year']}.json"
json_path.write_text(scorecard.model_dump_json(indent=2), encoding="utf-8")
print(f"\nSaved: {json_path}")
print(f"Score: {total_earned:.1f} / {total_possible:.1f}  ({percentage:.1f}%)")

### Rendered scorecard

In [ ]:
lines = [
    f"# Scorecard \u2014 {scorecard.subject} {scorecard.year}"
    + (f" ({scorecard.set_label})" if scorecard.set_label else ""),
    "",
    f"**{scorecard.total_points_earned:.1f} / {scorecard.total_points_possible:.1f}  ({scorecard.percentage:.1f}%)**",
    "",
]

if scorecard.review_flags:
    lines.append("> \u26a0\ufe0f **Review recommended:**")
    for f in scorecard.review_flags:
        lines.append(f"> - {f}")
    lines.append("")

for qs in scorecard.questions:
    lines.append(f"## Question {qs.question_id} \u2014 {qs.points_earned:.1f} / {qs.points_possible:.1f}")
    lines.append("")
    for ps in qs.point_scores:
        check = "\u2705" if ps.awarded else "\u274c"
        conf_icon = {"high": "", "medium": " \U0001f914", "low": " \u26a0\ufe0f"}.get(ps.grading_confidence, "")
        review_icon = " \U0001f50d" if ps.review_recommended else ""
        lines.append(f"- {check} **{ps.point_id}** ({ps.points_earned:.1f} pt){conf_icon}{review_icon}")
        lines.append(f"  - {ps.rationale}")
        if ps.transcript_evidence:
            lines.append(f"  - *Evidence:* `{ps.transcript_evidence}`")
    lines.append("")

md_text = "\n".join(lines)
md_path = CONFIG["output_dir"] / f"scorecard-{subj_slug}-{CONFIG['year']}.md"
md_path.write_text(md_text, encoding="utf-8")
print(f"Saved: {md_path}\n")

display(Markdown(md_text))